# DataHub Error Case 01 — Auth Failure: Invalid Token

**Monitor:** DataHub Auth Failure  
**Component:** datahub-gms

In [21]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

r = health_check()
print(f'GMS health: {r.status_code}')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
GMS health: 200


In [17]:
# Cell 2: Invalid token
INVALID_TOKEN = 'this.is.not.a.valid.jwt.token'
r = requests.get(
    f'{DATAHUB_URL}/api/graphql',
    headers=auth_headers(INVALID_TOKEN),
    json={'query': '{ __typename }'}
)
print(f'Status: {r.status_code}')
print(f'Response: {r.text[:300]}')

Status: 401
Response: {"timestamp":1781069110616,"status":401,"error":"Unauthorized","path":"/api/graphql"}


In [18]:
# Cell 3: Wrong credentials
r2 = requests.post(
    f'{DATAHUB_URL}/logIn',
    json={'username': 'wronguser', 'password': 'wrongpassword'}
)
print(f'Wrong credentials: {r2.status_code}')
print(f'Response: {r2.text[:200]}')

Wrong credentials: 401
Response: {"timestamp":1781069111275,"status":401,"error":"Unauthorized","path":"/logIn"}


In [19]:
# Cell 4: Find auth error logs
wait_for_logs(5)
print('Searching for auth errors (last 2 min)...')
hits = search_datahub_auth_errors(minutes_ago=2)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
Searching for auth errors (last 2 min)...
✅ Found 10 log entries:

  [2026-06-10T05:25:11] DEBUG | datahub-gms | c.d.a.a.AuthenticatorChain
         Caught exception while attempting to authenticate request using Authenticator com.datahub.authentication.authenticator.D

  [2026-06-10T05:25:11] DEBUG | datahub-gms | c.d.a.a.AuthenticatorChain
         Caught exception while attempting to authenticate request using Authenticator com.datahub.authentication.authenticator.D

  [2026-06-10T05:25:11] DEBUG | datahub-gms | c.d.a.a.AuthenticatorChain
         Caught exception while attempting to authenticate request using Authenticator com.datahub.authentication.authenticator.H

  [2026-06-10T05:25:11] DEBUG | datahub-gms | c.d.a.a.AuthenticatorChain
         Caught exception while attempting to authenticate request using Authenticator com.datahub.authentication.authenticator.D

  [2026-06-10T05:25:11] DEBUG | datahub-gms | c.d.a.a.AuthenticatorChain

In [22]:
# Cell 5: Fix
r_fix = get_token(DEFAULT_USER, DEFAULT_PASS)
print(f'Correct credentials: {r_fix.status_code}')
assert r_fix.status_code == 200

print('\n=== Summary ===')
print('Error:   401 Unauthorized')
print('Cause:   Invalid/expired token or wrong credentials')
print('Monitor: DataHub Auth Failure fires when count > 0 in 1 min')
print('Fix:     Re-login / refresh access token')

Correct credentials: 200

=== Summary ===
Error:   401 Unauthorized
Cause:   Invalid/expired token or wrong credentials
Monitor: DataHub Auth Failure fires when count > 0 in 1 min
Fix:     Re-login / refresh access token
